In [7]:
import keras
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from keras import layers

import numpy as np

In [8]:
#Load dataset
(x_train_tmp, y_train_tmp), (x_test_tmp, y_test_tmp) = keras.datasets.cifar100.load_data()
# print(y_train_tmp.shape)
x_train = []
y_train = []
x_test = []
y_test = []
for i in range (y_train_tmp.shape[0]):
    if (0 <= y_train_tmp[i][0] <= 19):
        x_train.append(x_train_tmp[i])
        y_train.append(y_train_tmp[i])
for i in range (y_test_tmp.shape[0]):
    if (0 <= y_test_tmp[i][0] <= 19):
        x_test.append(x_test_tmp[i])
        y_test.append(y_test_tmp[i])
x_train = np.array(x_train)
y_train = np.array(y_train)
x_test = np.array(x_test)
y_test = np.array(y_test)

x_train = x_train / 255.0
x_test = x_test / 255.0


trainY = to_categorical(y_train, num_classes = 20)
testY = to_categorical(y_test, num_classes = 20)

In [12]:
resnet50_model = keras.applications.ResNet50(
    include_top=False,
    weights="imagenet",
    input_tensor=None,
    input_shape=None,
    pooling=None,
)

# xception_model.summary()

model = keras.Sequential(
    [
        keras.Input(shape=(32, 32, 3)),
        resnet50_model,

        layers.Flatten(),

        layers.Dropout(0.4),
        layers.Dense(512),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        
        layers.Dropout(0.4),
        layers.Dense(256),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.Dense(20, activation='softmax')
    ]
)
model.summary()

94765736/94765736 [==============================] - 28s 0us/step
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 resnet50 (Functional)       (None, None, None, 2048   23587712  
                             )                                   
                                                                 
 flatten_1 (Flatten)         (None, 2048)              0         
                                                                 
 dropout_2 (Dropout)         (None, 2048)              0         
                                                                 
 dense_3 (Dense)             (None, 512)               1049088   
                                                                 
 batch_normalization_2 (Bat  (None, 512)               2048      
 chNormalization)                                                
                                                      

In [13]:
early_stopping = keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss=keras.losses.CategoricalCrossentropy(from_logits=False),
    metrics=['accuracy'],
)

model.fit(x_train, trainY, epochs=5, callbacks=[early_stopping], batch_size=32, validation_split=0.1)

Epoch 1/5
282/282 [==============================] - 399s 1s/step - loss: 2.2871 - accuracy: 0.3229 - val_loss: 3.2469 - val_accuracy: 0.0600
Epoch 2/5
282/282 [==============================] - 406s 1s/step - loss: 1.5630 - accuracy: 0.5364 - val_loss: 3.4154 - val_accuracy: 0.1070
Epoch 3/5
282/282 [==============================] - 407s 1s/step - loss: 1.2651 - accuracy: 0.6271 - val_loss: 2.3425 - val_accuracy: 0.3510
Epoch 4/5
282/282 [==============================] - 401s 1s/step - loss: 1.0657 - accuracy: 0.6870 - val_loss: 1.9580 - val_accuracy: 0.5060
Epoch 5/5
282/282 [==============================] - 405s 1s/step - loss: 0.9125 - accuracy: 0.7320 - val_loss: 1.6084 - val_accuracy: 0.5860


In [14]:
model.evaluate(x_test, testY)

63/63 [==============================] - 3s 44ms/step - loss: 1.6318 - accuracy: 0.5725


[1.6317765712738037, 0.5724999904632568]